# P4 Depth Completion — Kaggle Experiments

Batch evaluation of the repository's depth-completion methods (interpolation baselines, classical edge-preserving upsampling, SGNet deep branch, hybrid fusion) on:

- **HF NYU val** — `sayakpaul/nyu_depth_v2` (FastDepth 654-image labelled NYU test set, Apache-2.0), synthetic bicubic degradation, clean / noise / holes, ×2 / ×4 / ×8;
- **the official hackathon testset** `data/depthsr_testset/` (20 samples, native sensor-like missing depth, metres, ×2 / ×4 / ×8).

**Honesty protocol (read before quoting numbers):**

- Every table printed below comes from *this* run on *this* Kaggle GPU. Quote it as "Kaggle run, `<date>`, `<GPU>`" — it is **not** interchangeable with `docs/GPU_RUN_REPORT.md` (teammate RTX 5070), with local CPU smoke tests, or with the SGNet paper protocol (different data subset; see GPU_RUN_REPORT).
- A missing SGNet checkpoint makes `deep` / `hybrid*` report **not evaluated** with the reason (never silently dropped, never mislabelled).
- Metrics follow `docs/EVALUATION_METHODOLOGY.md`: RMSE / MAE / PSNR / SSIM / boundary RMSE / edge F1 over valid GT pixels, border 6 px, data_range 10 m; latency measured with `torch.cuda.synchronize()`.

**Requirements:** GPU runtime recommended (Session → Change accelerator). Internet ON *or* attach the HF tar / checkpoints as input datasets. Checkpoints are optional — baselines and classical always run.

In [ ]:
#@title 1. Environment report
import sys
import torch

print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    torch.cuda.synchronize()
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("gpu: none — runs will execute on CPU (slow but correct)")

In [ ]:
#@title 2. Clone repository and install dependencies
import os
import subprocess
import sys

REPO = "/kaggle/working/CV-Hackthon"
if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/KumarSrinidhi/CV-Hackthon.git", REPO],
        check=True,
    )
os.chdir(REPO)

# Install everything except torch/torchvision: Kaggle already ships a CUDA build,
# and letting pip resolve "torch" afresh can replace it with a CPU wheel.
reqs = [
    line.strip()
    for line in open("requirements.txt")
    if line.strip() and not line.startswith("#")
]
reqs = [r for r in reqs if not r.lower().startswith(("torch", "torchvision"))]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *reqs, "h5py", "huggingface_hub"],
    check=True,
)
sys.path.insert(0, REPO)
print("repo at", os.getcwd())

## Checkpoints (optional — required for `deep` and `hybrid*` numbers)

Checkpoints are intentionally **not** in git. To enable the deep and hybrid branches, attach a Kaggle dataset containing the official `SGNet_X4_R.pth` / `SGNet_X8_R.pth` / `SGNet_X16_R.pth` files (Add data → your dataset) and set its path below. The `.pth` files must sit **directly** in that folder, not in a subfolder. Leave `CKPT_DIR` empty to evaluate only the baselines and the classical branch.

In [ ]:
#@title 3. Checkpoint status
CKPT_DIR = ""  #@param {type:"string"}

import os

if CKPT_DIR.strip():
    os.environ["SGNET_CKPT_DIR"] = CKPT_DIR.strip()

from deep.settings import ckpt_dir
from deep.deep_model import resolve_checkpoint

print("checkpoint dir:", ckpt_dir())
for s in (2, 4, 8, 16):
    try:
        print(f"  x{s}: {resolve_checkpoint(s)}")
    except Exception as e:
        print(f"  x{s}: NOT AVAILABLE ({type(e).__name__}: {e})")

## Data A — HF NYU val tar (~955 MB)

One tar holds the FastDepth-labelled NYU val `.h5` samples used by every local HF run in `docs/`. With Internet ON the next cell downloads it from HuggingFace; otherwise attach a Kaggle dataset that contains `val-000000.tar` and set `TAR_PATH` below.

In [ ]:
#@title 4. Locate (or download) the HF val tar
TAR_PATH = ""  #@param {type:"string"}

import glob
import os

def _attached_tar():
    hits = glob.glob("/kaggle/input/**/val-*.tar", recursive=True)
    return hits[0] if hits else ""

if TAR_PATH.strip():
    TAR = TAR_PATH.strip()
elif _attached_tar():
    TAR = _attached_tar()
else:
    from huggingface_hub import HfApi, hf_hub_download

    files = HfApi().list_repo_files("sayakpaul/nyu_depth_v2", repo_type="dataset")
    cands = [f for f in files if f.endswith(".tar")]
    val_cands = [f for f in cands if "val" in os.path.basename(f).lower()] or cands
    TAR = hf_hub_download(
        "sayakpaul/nyu_depth_v2", repo_type="dataset", filename=sorted(val_cands)[0]
    )
print("using tar:", TAR, f"({os.path.getsize(TAR) / 1e6:.0f} MB)")

## Experiments A/B — HF NYU val, clean bicubic degradation

- **A (primary):** first 100 samples at ×4 — the same protocol as the local `val100` baselines (`scripts/eval_all3_hf.py`).
- **B:** 50 samples each at ×2 and ×8. Expected: `deep`/`hybrid*` at ×2 report *not evaluated* (SGNet has no ×2 scale); ×8 needs the X8 checkpoint.

Method list: `nearest bilinear bicubic classical deep hybrid hybrid_fixed02 hybrid_adaptive hybrid_selected_clean`.

In [ ]:
#@title 5. Experiment A — val100 ×4 clean (primary)
from scripts.eval_all3_hf import main as all3_main

all3_main(["--tars", TAR, "--max-samples", "100", "--scale", "4",
           "--out", "results/kaggle_hf_x4_clean100"])

In [ ]:
#@title 6. Experiment B — ×2 and ×8, 50 samples each
from scripts.eval_all3_hf import main as all3_main

for s in (2, 8):
    all3_main(["--tars", TAR, "--max-samples", "50", "--scale", str(s),
               "--out", f"results/kaggle_hf_x{s}_clean50"])

## Experiment C — HF NYU ×4 under noise and holes

Synthetic sensor noise (`noise_std=0.02` m) and random missing depth (`hole_ratio=0.10` of LR pixels), 50 samples each — the degraded-regime comparison from the evaluation methodology.

In [ ]:
#@title 7. Experiment C — ×4 noise + holes, 50 samples
from scripts.eval_all3_hf import HFNYUValDataset
from data.degradation import DegradationConfig
from eval.evaluate import evaluate_methods, save_results, summary_markdown

for tag, deg in (
    ("noisy", DegradationConfig(noise_std=0.02)),
    ("holes", DegradationConfig(hole_ratio=0.10)),
):
    ds = HFNYUValDataset(4, deg, tar_paths=[TAR])
    res = evaluate_methods(ds, ["bicubic", "classical", "deep", "hybrid"],
                           max_samples=50)
    save_results(res, f"results/kaggle_hf_x4_{tag}50", f"hf_x4_{tag}50")
    print(summary_markdown(res))

## Data B — official hackathon testset (native missing depth)

`data/depthsr_testset/` (branch `kumarbro`): 20 synthetic samples — 10 indoor rooms (~85 % valid LR) and 10 geometric-primitive scenes (~35–45 % valid) — with block-mean LR depth and sensor-like holes concentrated at depth discontinuities, values in metres.

This data is **not** re-degraded: methods receive the native LR depth + validity mask, so it directly exercises P4's missing-depth reconstruction. The native operator is block-mean, not the bicubic `S` that the consistency step assumes, so `hybrid` runs with `use_consistency=False` here (injected below); `hybrid_fixed02` / `hybrid_adaptive` already run without consistency.

In [ ]:
#@title 8. Extract the official testset from the kumarbro branch
import subprocess

subprocess.run(["git", "fetch", "-q", "origin", "kumarbro"], check=True)
subprocess.run(["git", "checkout", "-q", "FETCH_HEAD", "--", "data/depthsr_testset"],
               check=True)

import os
print(sorted(d for d in os.listdir("data/depthsr_testset")
             if os.path.isdir(os.path.join("data/depthsr_testset", d))))

In [ ]:
#@title 9. Experiment D — official testset, native missing depth, ×2/×4/×8
import os

import numpy as np
from PIL import Image

from data.dataset import DepthSRDataset
from data.preprocessing import to_float_rgb
from eval.evaluate import evaluate_methods, save_results, summary_markdown
from hybrid.pipeline import hybrid_hr_depth


class DepthSRTestset(DepthSRDataset):
    """Official 20-sample testset with its native (block-mean, sensor-hole) LR depth."""

    name = "depthsr_testset"
    unit = "m"
    data_range = 10.0

    def __init__(self, scale, root="data/depthsr_testset"):
        super().__init__(scale)
        self.folders = sorted(
            os.path.join(root, d) for d in os.listdir(root)
            if os.path.isdir(os.path.join(root, d))
        )

    def _num_raw(self):
        return len(self.folders)

    def _load_raw(self, i):
        d = self.folders[i]
        rgb = np.asarray(Image.open(os.path.join(d, "rgb.png")))
        gt = np.load(os.path.join(d, "depth_hr.npy")).astype(np.float32)
        return os.path.basename(d), to_float_rgb(rgb), gt, np.isfinite(gt) & (gt > 0)

    def __getitem__(self, i):
        # Serve the native LR depth/mask instead of synthetic re-degradation.
        name, rgb, gt, valid = self._load_raw(i)
        d = self.folders[i]
        lr = np.load(os.path.join(d, f"lr_depth_{self.scale}x.npy")).astype(np.float32)
        mask = np.load(os.path.join(d, f"lr_mask_{self.scale}x.npy")).astype(bool)
        return {
            "name": name,
            "hr_rgb": np.ascontiguousarray(rgb, dtype=np.float32),
            "depth_original": gt,
            "valid_mask_original": valid,
            "lr_depth": lr,
            "lr_valid_mask": mask,
            "lr_depth_clean": lr.copy(),
            "scale": self.scale,
            "unit": self.unit,
            "data_range": float(self.data_range),
            "operator_exact": False,
        }


# hybrid without consistency: the native LR operator is block-mean, not bicubic.
# The injected callable must honour the same honesty contract as the harness:
# no deep checkpoint -> NotImplementedError -> reported "not evaluated".
from deep.deep_model import resolve_checkpoint


def _hybrid_testset(lr, rgb, m, s=4):
    try:
        resolve_checkpoint(s)
    except Exception as e:
        raise NotImplementedError(f"deep checkpoint unavailable: {e}") from e
    return hybrid_hr_depth(lr, rgb, m, s, use_consistency=False)


testset_methods = {"hybrid": _hybrid_testset}

for s in (2, 4, 8):
    ds = DepthSRTestset(s)
    res = evaluate_methods(
        ds,
        ["nearest", "bilinear", "bicubic", "classical", "deep",
         "hybrid", "hybrid_fixed02", "hybrid_adaptive"],
        methods=testset_methods,
    )
    save_results(res, f"results/kaggle_testset_x{s}", f"testset_x{s}")
    print(summary_markdown(res))

## Results

Every experiment writes to `/kaggle/working/CV-Hackthon/results/kaggle_*`:

- `<tag>_summary.md` — the table printed above (also lists *not evaluated* methods with reasons);
- `<tag>_summary.json` — machine-readable summary + metadata (dataset, scale, degradation, border);
- `<tag>_per_image.csv` — per-sample metrics + latency.

Collect them with Kaggle's **Download output** (right sidebar) or publish the notebook. The cell below re-prints every summary table produced in this run.

In [ ]:
#@title 10. Print all summary tables from this run
from IPython.display import Markdown, display
import glob

for md_path in sorted(glob.glob("results/kaggle_*/*summary.md")):
    print(f"--- {md_path} ---")
    display(Markdown(open(md_path).read()))

## Reproduction commands (equivalent CLI)

```bash
python scripts/eval_all3_hf.py --tars <val.tar> --max-samples 100 --scale 4 --out results/kaggle_hf_x4_clean100
python -m eval.benchmark --dataset nyu --split val --scales 2 4 8 \
    --methods nearest bilinear bicubic classical deep hybrid_fixed02 hybrid_adaptive hybrid_selected_clean \
    --out results/benchmark_val --runs 10
```

Protocol details: `docs/EVALUATION_METHODOLOGY.md`. Prior GPU numbers: `docs/GPU_RUN_REPORT.md` (never re-labelled by this notebook).